# DynaGuard Inference Pipeline
### 100-Turn Compliance Evaluation via `dynaguard_chat_messages`
**Kaggle & Google Colab Compatible**

A lightweight, standalone, and transparent inference notebook for evaluating **DynaGuard** (8B / 4B / 1.7B) guardian models on structured chat message test cases in `dynaguard_chat_messages/`.

**Key Pipeline Principles:**
- **Exact Official Chat Template**: Passes the multi-turn dialogue message array directly to `tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, ...)` as demonstrated in the DynaGuard documentation.
- **Sequential Processing**: Evaluates test cases one by one directly from `dynaguard_chat_messages/` without accumulating activations in GPU memory.
- **Critical Transparent Decoding**: Slices only newly generated tokens (`outputs[0][inputs['input_ids'].shape[-1]:]`) and preserves all raw model tokens verbatim without parsing, stripping XML, or modifying text.
- **Resume & Reproducibility**: Skips existing outputs on re-run, saves sent prompts to `sent_prompts/`, and logs execution metadata to `logs/run_log.jsonl`.

## 1. Environment Detection & Dependencies

In [ ]:
import os, sys, json, re, time, datetime

# Detect execution environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

platform_name = 'Google Colab' if IS_COLAB else ('Kaggle' if IS_KAGGLE else 'Local / Standalone')
print(f"Detected Platform: {platform_name}")

# In cloud environments (Colab & Kaggle), ensure repository test cases are available
if (IS_COLAB or IS_KAGGLE) and not os.path.exists('dynaguard_chat_messages'):
    if not os.path.exists('DynaGuard'):
        print("Cloning DynaGuard repository from GitHub...")
        !git clone https://github.com/AkarshiAaryan/DynaGuard.git
    
    # Ensure dynaguard_chat_messages is accessible in the working directory
    if os.path.exists('DynaGuard/dynaguard_chat_messages') and not os.path.exists('dynaguard_chat_messages'):
        import shutil
        try:
            os.symlink('DynaGuard/dynaguard_chat_messages', 'dynaguard_chat_messages')
        except Exception:
            shutil.copytree('DynaGuard/dynaguard_chat_messages', 'dynaguard_chat_messages')
        print("Linked dynaguard_chat_messages to working directory.")

# Install required dependencies
!pip install -q -U transformers accelerate


## 2. Hardware Detection & Device Setup

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f"Detected {gpu_count} CUDA GPU(s):")
    for i in range(gpu_count):
        vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  - GPU {i}: {torch.cuda.get_device_name(i)} ({vram_gb:.1f} GB VRAM)")
    print(f"bfloat16 supported: {bf16_supported}")
    # Use bfloat16 on Ampere/Hopper (A100, L4) or float16 on Turing (T4)
    TORCH_DTYPE = torch.bfloat16 if bf16_supported else torch.float16
    DEVICE_MAP = 'auto'
    
    # Balanced memory mapping: reserves headroom on each GPU to avoid OOM during loading
    MAX_MEMORY = {
        i: f"{max(1, int((torch.cuda.get_device_properties(i).total_memory / (1024**3)) * 0.82))}GiB"
        for i in range(gpu_count)
    }
    MAX_MEMORY['cpu'] = '24GiB'
    print(f"Balanced Max Memory Mapping: {MAX_MEMORY}")
else:
    print("No CUDA GPU detected. Falling back to CPU.")
    TORCH_DTYPE = torch.float32
    DEVICE_MAP = None
    MAX_MEMORY = None

print(f"Selected Precision: {TORCH_DTYPE}, Device Map: {DEVICE_MAP}")


## 3. Directory & Input Configuration
Uses `dynaguard_chat_messages/` containing the 100 benchmark test cases structured as JSON chat message arrays.

In [ ]:
# Determine working base directory
if IS_KAGGLE:
    BASE_DIR = '/kaggle/working'
elif IS_COLAB:
    BASE_DIR = os.getcwd()
else:
    BASE_DIR = '.'

# Comprehensive search for test cases across local, cloned, and Kaggle input paths
candidates = [
    'dynaguard_chat_messages',
    os.path.join(BASE_DIR, 'dynaguard_chat_messages'),
    os.path.join(BASE_DIR, 'DynaGuard', 'dynaguard_chat_messages'),
    'DynaGuard/dynaguard_chat_messages',
    'dynaguard_inputs',
    os.path.join(BASE_DIR, 'dynaguard_inputs'),
    os.path.join(BASE_DIR, 'DynaGuard', 'dynaguard_inputs'),
    'DynaGuard/dynaguard_inputs',
]

# Search Kaggle dataset input paths if attached
if os.path.exists('/kaggle/input'):
    for root, dirs, _ in os.walk('/kaggle/input'):
        for target in ['dynaguard_chat_messages', 'dynaguard_inputs']:
            if target in dirs:
                candidates.append(os.path.join(root, target))

INPUT_DIR = next((d for d in candidates if os.path.isdir(d)), None)

# Auto-recovery if not found yet in cloud environment
if INPUT_DIR is None and (IS_COLAB or IS_KAGGLE):
    print("Test cases not found locally. Cloning repository from GitHub...")
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/AkarshiAaryan/DynaGuard.git"], check=False)
    for c in [
        'dynaguard_chat_messages',
        'DynaGuard/dynaguard_chat_messages',
        os.path.join(BASE_DIR, 'DynaGuard', 'dynaguard_chat_messages'),
        'dynaguard_inputs',
        os.path.join(BASE_DIR, 'DynaGuard', 'dynaguard_inputs'),
    ]:
        if os.path.isdir(c):
            INPUT_DIR = c
            break

if INPUT_DIR is None or not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(
        "Could not find test cases directory ('dynaguard_chat_messages' or 'dynaguard_inputs').\n\n"
        "If you are running on Kaggle:\n"
        "  1. Ensure Internet is enabled: Go to Session options (right panel) or Settings (top menu) -> Internet -> ON.\n"
        "  2. Re-run Cell 1 to clone the repository with the test cases."
    )

OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
SENT_PROMPTS_DIR = os.path.join(BASE_DIR, 'sent_prompts')
LOGS_DIR = os.path.join(BASE_DIR, 'logs')
RUN_LOG_PATH = os.path.join(LOGS_DIR, 'run_log.jsonl')

for d in [OUTPUT_DIR, SENT_PROMPTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Model selection:
# - 'tomg-group-umd/DynaGuard-4B' (Fast & low-memory, ideal for 15GB T4 GPUs)
# - 'tomg-group-umd/DynaGuard-8B' (Flagship 8B guardian model)
# - 'tomg-group-umd/DynaGuard-1.7B' (Ultra-lightweight)
MODEL_ID = os.environ.get('DYNAGUARD_MODEL_ID', 'tomg-group-umd/DynaGuard-4B')

# Inference mode: 'fast' (<answer>) or 'reasoning' / 'cot' (<think>)
INFERENCE_MODE = os.environ.get('DYNAGUARD_INFERENCE_MODE', 'fast')

def extract_policy_and_dialogue(content):
    """Extract policy rules and dialogue from either a list of chat message dicts or a raw string."""
    if isinstance(content, list):
        if not content:
            return '', ''
        first_content = content[0].get('content', '')
        if 'Policy:' in first_content:
            parts = first_content.split('\n\n', 1)
            policy = parts[0].replace('Policy:', '').strip()
            user_start = parts[1].strip() if len(parts) > 1 else ''
        else:
            policy = ''
            user_start = first_content

        dialogue_lines = []
        if user_start:
            dialogue_lines.append(f'User: {user_start}')

        for turn in content[1:]:
            role = 'User' if turn.get('role', '').lower() == 'user' else 'Agent'
            dialogue_lines.append(f"{role}: {turn.get('content', '')}")

        return policy, '\n'.join(dialogue_lines)
    else:
        text = str(content)
        if 'Dialogue:' in text:
            parts = text.split('Dialogue:', 1)
            return parts[0].replace('Policy:', '').strip(), parts[1].strip()
        elif 'Policy:' in text:
            parts = text.split('\n\n', 1)
            return parts[0].replace('Policy:', '').strip(), (parts[1].strip() if len(parts) > 1 else '')
        return '', text

def build_inference_prompt(test_case_content, mode='fast'):
    """Construct the exact prompt format required by DynaGuard."""
    policy, dialogue = extract_policy_and_dialogue(test_case_content)
    if mode.lower() in ('reasoning', 'cot'):
        instruction = 'Evaluate the following dialogue for compliance with the given policy. First, think through your reasoning step-by-step. Then, provide the final answer.'
        suffix = '<think>\n'
    else:
        instruction = 'Evaluate the following dialogue for compliance with the given policy. Provide the final answer directly.'
        suffix = '<answer>\n'

    return f"""[INST]
{instruction}

## Policy
{policy}

## Dialogue
{dialogue}
[/INST]
{suffix}"""

MAX_NEW_TOKENS = 100  # Cap generation length so model doesn't run on endlessly
TEMPERATURE = 0.1     # Low temperature sampling breaks repetitive greedy degeneration loops
TOP_P = 0.95
DO_SAMPLE = True
REPETITION_PENALTY = 1.15  # Prevents repeating the exact same sentence again and again

print(f"Active Model ID  : {MODEL_ID}")
print(f"Inference Mode   : {INFERENCE_MODE}")
print(f"Input Directory  : {INPUT_DIR} (exists: {os.path.isdir(INPUT_DIR)})")
print(f"Output Directory : {OUTPUT_DIR}")


## 4. Prompt Analysis: Inspecting the DynaGuard Chat Message Format
Examines the structure of `TC01_student_own_grade.json` and previews the raw message array.

In [ ]:
sample_case_path = os.path.join(INPUT_DIR, 'TC01_student_own_grade.json')
if os.path.exists(sample_case_path):
    with open(sample_case_path, 'r', encoding='utf-8') as f:
        sample_content = json.load(f)
    
    sample_prompt = build_inference_prompt(sample_content, mode=INFERENCE_MODE)
    print("=" * 75)
    print(f"SAMPLE DYNAGUARD PROMPT PREVIEW (TC01 - Mode: {INFERENCE_MODE}):")
    print("=" * 75)
    print(sample_prompt)
    print("=" * 75)
else:
    print(f"Sample file not found at {sample_case_path}")


## 5. Load DynaGuard Model & Tokenizer

In [ ]:
import gc
# Clean up leftover allocations from any previous crashed cells
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading model and tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model_kwargs = {
    'torch_dtype': TORCH_DTYPE,
    'trust_remote_code': True,
    'low_cpu_mem_usage': True,
}
if DEVICE_MAP is not None:
    model_kwargs['device_map'] = DEVICE_MAP
if MAX_MEMORY is not None:
    model_kwargs['max_memory'] = MAX_MEMORY

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
print("Model loaded successfully!")


## 6. Single-Case Verification Test
Runs `TC01_student_own_grade.json` through the model to observe the raw verdict.

In [ ]:
if os.path.exists(sample_case_path):
    prompt_text = build_inference_prompt(sample_content, mode=INFERENCE_MODE)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    
    start_t = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE if DO_SAMPLE else None,
            do_sample=DO_SAMPLE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start_t
    
    prompt_length = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=False)
    print(f"Inference Time: {elapsed:.2f}s")
    print("Raw Model Generated Response:")
    print("-" * 50)
    print(response)
    print("-" * 50)


## 7. Sequential Batch Evaluation Over All 100 Test Cases
Processes each test case in `dynaguard_chat_messages/` one-by-one in natural numerical order (`TC01` to `TC100`), writes raw outputs verbatim, and logs metrics.

In [ ]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

if not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(f"Input directory '{INPUT_DIR}' does not exist. Please run Section 3 first.")

# Collect all test cases (.json files representing message arrays or .txt files)
test_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.json') and f != 'dynaguard_messages.json']
if not test_files:
    test_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]

test_files.sort(key=natural_sort_key)
total_cases = len(test_files)
print(f"Found {total_cases} test case(s) in {INPUT_DIR}.")
if total_cases > 0:
    print(f"Range: {test_files[0]}  -->  {test_files[-1]}")

RERUN_EXISTING = False  # Set to True to re-evaluate already completed cases

successful_count = 0
failed_count = 0
skipped_count = 0

for idx, filename in enumerate(test_files, start=1):
    stem = os.path.splitext(filename)[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_output.txt")
    prompt_path = os.path.join(SENT_PROMPTS_DIR, f"{stem}_prompt.txt")
    in_path = os.path.join(INPUT_DIR, filename)

    # Resume logic: skip existing non-empty outputs
    if not RERUN_EXISTING and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"[{idx:03d}/{total_cases:03d}] {filename} -> SKIPPED (already evaluated)")
        skipped_count += 1
        continue

    print(f"[{idx:03d}/{total_cases:03d}] Processing {filename} ... ", end='', flush=True)

    try:
        # Load test case content
        if filename.endswith('.json'):
            with open(in_path, 'r', encoding='utf-8') as f:
                raw_content = json.load(f)
        else:
            with open(in_path, 'r', encoding='utf-8') as f:
                raw_content = f.read()

        # Build prompt using DynaGuard [INST] format
        full_prompt = build_inference_prompt(raw_content, mode=INFERENCE_MODE)
        inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

        # Archive verbatim prompt for reproducibility
        with open(prompt_path, 'w', encoding='utf-8') as pf:
            pf.write(full_prompt)

        # Execute generation
        prompt_length = inputs["input_ids"].shape[-1]
        start_t = time.perf_counter()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE if DO_SAMPLE else None,
                do_sample=DO_SAMPLE,
                top_p=TOP_P,
                repetition_penalty=REPETITION_PENALTY,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        elapsed_t = time.perf_counter() - start_t

        # CRITICAL: Decode only newly generated tokens verbatim
        raw_generated = tokenizer.decode(
            outputs[0][prompt_length:],
            skip_special_tokens=False,
        )

        # Save raw model response without any parsing
        with open(out_path, 'w', encoding='utf-8') as of:
            of.write(raw_generated)

        # Log record
        log_entry = {
            'input_filename': filename,
            'output_filename': f"{stem}_output.txt",
            'status': 'success',
            'inference_time': round(elapsed_t, 4),
            'error_message': None,
            'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        }
        with open(RUN_LOG_PATH, 'a', encoding='utf-8') as lf:
            lf.write(json.dumps(log_entry) + '\n')

        print(f"DONE ({elapsed_t:.2f}s)")
        successful_count += 1

    except Exception as exc:
        print(f"FAILED: {exc}")
        failed_count += 1
        log_entry = {
            'input_filename': filename,
            'output_filename': f"{stem}_output.txt",
            'status': 'failed',
            'inference_time': 0.0,
            'error_message': str(exc),
            'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        }
        with open(RUN_LOG_PATH, 'a', encoding='utf-8') as lf:
            lf.write(json.dumps(log_entry) + '\n')

print("\n" + "=" * 70)
print(f"Inference Run Complete: {successful_count} succeeded, {failed_count} failed, {skipped_count} skipped.")
print(f"Outputs saved to: {OUTPUT_DIR}")
print(f"Sent prompts to : {SENT_PROMPTS_DIR}")
print(f"Log saved to    : {RUN_LOG_PATH}")
print("=" * 70)


## 8. Summary Inspection

In [ ]:
import pandas as pd

# Load and display run log records
if os.path.exists(RUN_LOG_PATH):
    records = []
    with open(RUN_LOG_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    
    df = pd.DataFrame(records)
    print(f"Total logged inferences: {len(df)}")
    display(df.tail(20))
else:
    print("No log file found yet.")

# Preview first 3 raw outputs
out_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('_output.txt')], key=natural_sort_key)
print(f"\nTotal output files saved: {len(out_files)}")
for f in out_files[:3]:
    print(f"\n=== {f} ===")
    with open(os.path.join(OUTPUT_DIR, f), 'r', encoding='utf-8') as fh:
        print(fh.read())
    print("=" * 40)